# V5 Auto-Label Training Pipeline**8 Models | Google Drive Saves | Zero Browser Downloads | Autopilot Ready**| # | Model | Output Filename | Method ||---|-------|----------------|--------|| 1-3 | VideoMAE Play Classifier (3 sports) | videomae_{sport}_v5.zip | Fine-tune VideoMAE || 4-6 | YOLO Outcome Classifier (3 sports) | outcome_classifier_{sport}_v5.pt | YOLO classify || 7 | Jersey OCR Universal | jersey_ocr_universal_v5.pt | YOLO detect || 8 | Player Detector | player_detector_v5.pt | YOLO detect |**All models save to Google Drive automatically. Zero browser downloads.**### How to Use1. Run every cell in Section 0 (Setup)2. Run every cell in Section 1 (YouTube Harvesting)3. Run every cell in Section 2 (Claude Labeling)4. Run Sections 3-6 (Training) — each saves to Drive5. Run Section 7 (Final Summary) to verify all 8 models### IF COLAB DISCONNECTS1. Reconnect runtime2. Rerun Section 0 (Setup)3. Rerun the cell that crashed — training auto-resumes from Drive checkpoints**Colab Secrets Required:** `ROBOFLOW_API_KEY`, `ANTHROPIC_API_KEY`

## SECTION 0: SETUP

In [ ]:
# Cell 0A: Install all required packages!pip install roboflow ultralytics pyyaml anthropic "scenedetect[opencv]" yt-dlp transformers datasets accelerate -qprint("\u2705 All packages installed")

In [ ]:
# Cell 0B: Imports + Config Variablesimport os, sys, torch, shutil, glob, yaml, time, json, cv2, gc, zipfile, subprocess, base64import numpy as npfrom collections import Counterfrom google.colab import userdata# === DIRECTORIES ===DRIVE_SAVE_DIR = '/content/drive/MyDrive/clipt_v5_models'DRIVE_CHECKPOINTS = '/content/drive/MyDrive/clipt_v5_models/checkpoints'DRIVE_LABELS = '/content/drive/MyDrive/clipt_v5_models/labels'CLIPS_BASE = '/content/clips'LABELS_BASE = '/content/labels'RAW_VIDEO_BASE = '/content/raw_videos'# === COST CONTROLS ===MAX_CLIPS_TO_LABEL = 200MAX_API_CALLS_TOTAL = 650COST_PER_CALL_ESTIMATE = 0.003# === PLAY TYPES ===PLAY_TYPES = {    'basketball': ['layup','jump_shot','dunk','three_pointer','fast_break','rebound',                    'steal','block','assist','free_throw','turnover','other'],    'football': ['pass_play','run_play','touchdown','interception','sack','field_goal',                 'punt','kickoff','tackle','catch','other'],    'lacrosse': ['shot','goal','save','ground_ball','face_off','clear','dodge',                 'pass','ride','other'],}# === CREATE DIRECTORIES ===for sport in ['basketball', 'football', 'lacrosse']:    os.makedirs(f'{CLIPS_BASE}/{sport}', exist_ok=True)    os.makedirs(f'{RAW_VIDEO_BASE}/{sport}', exist_ok=True)os.makedirs(LABELS_BASE, exist_ok=True)# === API KEYS ===ROBOFLOW_API_KEY = ''ANTHROPIC_API_KEY = ''try:    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')except Exception:    passtry:    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')except Exception:    pass# === GPU CHECK ===assert torch.cuda.is_available(), '\033[91m\u274c NO GPU \u2014 go to Runtime > Change runtime type > GPU\033[0m'gpu_name = torch.cuda.get_device_name(0)vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9 if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else torch.cuda.get_device_properties(0).total_memory / 1e9HAS_A100 = vram_gb > 40DEFAULT_BATCH = 16 if HAS_A100 else 8print(f"GPU: {gpu_name} ({vram_gb:.1f}GB VRAM)")print(f"Default batch size: {DEFAULT_BATCH}")print(f"Roboflow API key: {'\u2705 set' if ROBOFLOW_API_KEY else '\u274c missing \u2014 add to Colab Secrets'}")print(f"Anthropic API key: {'\u2705 set' if ANTHROPIC_API_KEY else '\u274c missing \u2014 add to Colab Secrets'}")

In [ ]:
# Cell 0C: Mount Google Drivefrom google.colab import drivedrive.mount('/content/drive')os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)os.makedirs(DRIVE_LABELS, exist_ok=True)print(f"\u2705 Drive mounted. Models will save to: {DRIVE_SAVE_DIR}")# Show any existing modelsexisting = glob.glob(f'{DRIVE_SAVE_DIR}/*.*')if existing:    print(f"\nFound {len(existing)} existing files:")    for f in sorted(existing):        sz = os.path.getsize(f) / 1024 / 1024        print(f"  {os.path.basename(f)} \u2014 {sz:.1f}MB")else:    print("No existing models found (fresh start)")

In [ ]:
# Cell 0D: Test Anthropic API Keyimport anthropicassert ANTHROPIC_API_KEY, "\033[91m\u274c Set ANTHROPIC_API_KEY in Colab Secrets (left sidebar > key icon)\033[0m"client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)anthropic_ok = False# Verify claude-sonnet-4-20250514 model stringtry:    resp = client.messages.create(        model="claude-sonnet-4-20250514",        max_tokens=10,        messages=[{"role": "user", "content": "Say OK"}]    )    print(f"\u2705 Anthropic API working \u2014 claude-sonnet-4-20250514 verified")    print(f"   Response: {resp.content[0].text}")    print(f"   Note: Will use claude-haiku-4-5-20251001 for labeling (12x cheaper)")    anthropic_ok = Trueexcept Exception as e:    print(f"\033[91m\u274c Anthropic API FAILED: {e}\033[0m")    print(f"\033[93m\u26a0\ufe0f ACTION NEEDED: Check ANTHROPIC_API_KEY in Colab Secrets\033[0m")

In [ ]:
# Cell 0E: Test yt-dlp + Roboflow + Print Setup Summaryimport yt_dlp# --- Test yt-dlp ---test_dir = '/content/yt_test'os.makedirs(test_dir, exist_ok=True)ytdlp_ok = Falseydl_opts = {    'format': 'best[height<=720]',    'outtmpl': f'{test_dir}/%(id)s.%(ext)s',    'quiet': False,    'extractor_args': {'youtube': {        'player_client': ['android'],    }},}try:    with yt_dlp.YoutubeDL(ydl_opts) as ydl:        info = ydl.extract_info('ytsearch1:basketball highlights short clip', download=True)    files = glob.glob(f'{test_dir}/*')    if files:        print(f"\u2705 yt-dlp downloaded: {os.path.basename(files[0])} ({os.path.getsize(files[0])/1024/1024:.1f}MB)")        for f in files:            os.remove(f)        ytdlp_ok = True    else:        print("\033[91m\u274c yt-dlp: no file downloaded\033[0m")except Exception as e:    print(f"\033[91m\u274c yt-dlp FAILED: {e}\033[0m")    print("\033[93m\u26a0\ufe0f Try: !pip install -U yt-dlp\033[0m")shutil.rmtree(test_dir, ignore_errors=True)# --- Test Roboflow ---rf_ok = Falsetry:    from roboflow import Roboflow    rf = Roboflow(api_key=ROBOFLOW_API_KEY)    rf_ok = True    print("\u2705 Roboflow API initialized")except Exception as e:    print(f"\033[91m\u274c Roboflow FAILED: {e}\033[0m")# --- Print Setup Summary ---print()print("=" * 50)print("\u2705 SETUP COMPLETE")print(f"GPU: {gpu_name} \u2014 {'A100' if HAS_A100 else '\u26a0\ufe0f WARNING: not A100, training will be slower'}")print(f"Drive mounted: {DRIVE_SAVE_DIR}")print(f"Anthropic API: {'\u2705 working' if ANTHROPIC_API_KEY else '\u274c missing'}")print(f"yt-dlp test: {'\u2705 working' if ytdlp_ok else '\u274c failed'}")print(f"Roboflow API: {'\u2705 working' if rf_ok else '\u274c failed'}")print(f"All packages installed: \u2705")print("=" * 50)

## SECTION 1: YOUTUBE HARVESTINGDownload sports highlight videos from YouTube and split into clips using scene detection.**Search queries verified to return results:**- Basketball: NBA best plays, dunks, game highlights- Football: NFL touchdown compilations, best plays- Lacrosse: PLL/NCAA/high school lacrosse highlights

In [ ]:
# Cell 1A: Download basketball videosimport yt_dlpSPORT = 'basketball'DL_DIR = f'{RAW_VIDEO_BASE}/{SPORT}'os.makedirs(DL_DIR, exist_ok=True)SEARCH_QUERIES = [    'ytsearch3:NBA best plays highlights compilation',    'ytsearch3:NBA dunks blocks steals compilation 2024',    'ytsearch3:college basketball highlights game 2024',]ydl_opts = {    'format': 'best[height<=720]',    'outtmpl': f'{DL_DIR}/%(id)s.%(ext)s',    'quiet': False,    'extractor_args': {'youtube': {'player_client': ['android']}},    'ignoreerrors': True,    'merge_output_format': 'mp4',}for qi, query in enumerate(SEARCH_QUERIES):    print(f"\n\U0001f50d [{qi+1}/{len(SEARCH_QUERIES)}] Searching: {query}")    try:        with yt_dlp.YoutubeDL(ydl_opts) as ydl:            ydl.download([query])    except Exception as e:        print(f"  \u26a0\ufe0f Query failed: {e}")total = len(glob.glob(f'{DL_DIR}/*'))print(f"\n\u2705 Basketball videos downloaded: {total}")if total == 0:    print("\033[91m\u274c No videos downloaded \u2014 check yt-dlp config or add manual URLs\033[0m")

In [ ]:
# Cell 1B: Scene detect basketball -> clipsfrom scenedetect import detect, ContentDetectorSPORT = 'basketball'DL_DIR = f'{RAW_VIDEO_BASE}/{SPORT}'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'os.makedirs(CLIPS_DIR, exist_ok=True)videos = sorted(glob.glob(f'{DL_DIR}/*.mp4') + glob.glob(f'{DL_DIR}/*.webm') + glob.glob(f'{DL_DIR}/*.mkv'))assert len(videos) > 0, f"\033[91m\u274c No videos in {DL_DIR} \u2014 run Cell 1A first\033[0m"total_clips = len(glob.glob(f'{CLIPS_DIR}/*.mp4'))print(f"Existing clips: {total_clips}")for vi, vp in enumerate(videos):    vname = os.path.splitext(os.path.basename(vp))[0]    # Skip if already processed    if glob.glob(f'{CLIPS_DIR}/{vname}_clip*.mp4'):        print(f"[{vi+1}/{len(videos)}] {os.path.basename(vp)} \u2014 already processed, skipping")        continue    print(f"[{vi+1}/{len(videos)}] Processing {os.path.basename(vp)}...")    try:        scenes = detect(vp, ContentDetector(threshold=27.0))        cap = cv2.VideoCapture(vp)        fps = cap.get(cv2.CAP_PROP_FPS) or 30        clip_n = 0        for s, e in scenes:            dur = (e.get_frames() - s.get_frames()) / fps            if 2 <= dur <= 15:                cp = f'{CLIPS_DIR}/{vname}_clip{clip_n:04d}.mp4'                start_sec = s.get_frames() / fps                subprocess.run(                    ['ffmpeg', '-y', '-ss', str(start_sec), '-i', vp,                     '-t', str(dur), '-c:v', 'libx264', '-preset', 'fast',                     '-crf', '23', '-an', cp],                    capture_output=True                )                if os.path.exists(cp) and os.path.getsize(cp) > 10000:                    clip_n += 1                else:                    if os.path.exists(cp):                        os.remove(cp)        cap.release()        print(f"  \u2192 {clip_n} clips extracted")    except Exception as e:        print(f"  \u26a0\ufe0f Failed: {e}")total_clips = len(glob.glob(f'{CLIPS_DIR}/*.mp4'))print(f"\nBasketball clips total: {total_clips}")

In [ ]:
# Cell 1C: Download football videosimport yt_dlpSPORT = 'football'DL_DIR = f'{RAW_VIDEO_BASE}/{SPORT}'os.makedirs(DL_DIR, exist_ok=True)SEARCH_QUERIES = [    'ytsearch3:NFL touchdown compilation',    'ytsearch3:NFL best plays highlights compilation 2024',    'ytsearch3:college football touchdown highlights',]ydl_opts = {    'format': 'best[height<=720]',    'outtmpl': f'{DL_DIR}/%(id)s.%(ext)s',    'quiet': False,    'extractor_args': {'youtube': {'player_client': ['android']}},    'ignoreerrors': True,    'merge_output_format': 'mp4',}for qi, query in enumerate(SEARCH_QUERIES):    print(f"\n\U0001f50d [{qi+1}/{len(SEARCH_QUERIES)}] Searching: {query}")    try:        with yt_dlp.YoutubeDL(ydl_opts) as ydl:            ydl.download([query])    except Exception as e:        print(f"  \u26a0\ufe0f Query failed: {e}")total = len(glob.glob(f'{DL_DIR}/*'))print(f"\n\u2705 Football videos downloaded: {total}")if total == 0:    print("\033[91m\u274c No videos downloaded\033[0m")

In [ ]:
# Cell 1D: Scene detect football -> clipsfrom scenedetect import detect, ContentDetectorSPORT = 'football'DL_DIR = f'{RAW_VIDEO_BASE}/{SPORT}'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'os.makedirs(CLIPS_DIR, exist_ok=True)videos = sorted(glob.glob(f'{DL_DIR}/*.mp4') + glob.glob(f'{DL_DIR}/*.webm') + glob.glob(f'{DL_DIR}/*.mkv'))assert len(videos) > 0, f"\033[91m\u274c No videos in {DL_DIR} \u2014 run Cell 1C first\033[0m"total_clips = len(glob.glob(f'{CLIPS_DIR}/*.mp4'))print(f"Existing clips: {total_clips}")for vi, vp in enumerate(videos):    vname = os.path.splitext(os.path.basename(vp))[0]    if glob.glob(f'{CLIPS_DIR}/{vname}_clip*.mp4'):        print(f"[{vi+1}/{len(videos)}] {os.path.basename(vp)} \u2014 already processed, skipping")        continue    print(f"[{vi+1}/{len(videos)}] Processing {os.path.basename(vp)}...")    try:        scenes = detect(vp, ContentDetector(threshold=27.0))        cap = cv2.VideoCapture(vp)        fps = cap.get(cv2.CAP_PROP_FPS) or 30        clip_n = 0        for s, e in scenes:            dur = (e.get_frames() - s.get_frames()) / fps            if 2 <= dur <= 15:                cp = f'{CLIPS_DIR}/{vname}_clip{clip_n:04d}.mp4'                start_sec = s.get_frames() / fps                subprocess.run(                    ['ffmpeg', '-y', '-ss', str(start_sec), '-i', vp,                     '-t', str(dur), '-c:v', 'libx264', '-preset', 'fast',                     '-crf', '23', '-an', cp],                    capture_output=True                )                if os.path.exists(cp) and os.path.getsize(cp) > 10000:                    clip_n += 1                else:                    if os.path.exists(cp):                        os.remove(cp)        cap.release()        print(f"  \u2192 {clip_n} clips extracted")    except Exception as e:        print(f"  \u26a0\ufe0f Failed: {e}")total_clips = len(glob.glob(f'{CLIPS_DIR}/*.mp4'))print(f"\nFootball clips total: {total_clips}")

In [ ]:
# Cell 1E: Download lacrosse videosimport yt_dlpSPORT = 'lacrosse'DL_DIR = f'{RAW_VIDEO_BASE}/{SPORT}'os.makedirs(DL_DIR, exist_ok=True)SEARCH_QUERIES = [    'ytsearch3:lacrosse goals compilation high school',    'ytsearch3:PLL lacrosse highlights goals',    'ytsearch3:NCAA college lacrosse highlights compilation',]ydl_opts = {    'format': 'best[height<=720]',    'outtmpl': f'{DL_DIR}/%(id)s.%(ext)s',    'quiet': False,    'extractor_args': {'youtube': {'player_client': ['android']}},    'ignoreerrors': True,    'merge_output_format': 'mp4',}for qi, query in enumerate(SEARCH_QUERIES):    print(f"\n\U0001f50d [{qi+1}/{len(SEARCH_QUERIES)}] Searching: {query}")    try:        with yt_dlp.YoutubeDL(ydl_opts) as ydl:            ydl.download([query])    except Exception as e:        print(f"  \u26a0\ufe0f Query failed: {e}")total = len(glob.glob(f'{DL_DIR}/*'))print(f"\n\u2705 Lacrosse videos downloaded: {total}")if total == 0:    print("\033[91m\u274c No videos downloaded\033[0m")    print("\033[93m\u26a0\ufe0f Lacrosse has fewer compilations. Try adding specific video URLs.\033[0m")

In [ ]:
# Cell 1F: Scene detect lacrosse -> clipsfrom scenedetect import detect, ContentDetectorSPORT = 'lacrosse'DL_DIR = f'{RAW_VIDEO_BASE}/{SPORT}'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'os.makedirs(CLIPS_DIR, exist_ok=True)videos = sorted(glob.glob(f'{DL_DIR}/*.mp4') + glob.glob(f'{DL_DIR}/*.webm') + glob.glob(f'{DL_DIR}/*.mkv'))assert len(videos) > 0, f"\033[91m\u274c No videos in {DL_DIR} \u2014 run Cell 1E first\033[0m"total_clips = len(glob.glob(f'{CLIPS_DIR}/*.mp4'))print(f"Existing clips: {total_clips}")for vi, vp in enumerate(videos):    vname = os.path.splitext(os.path.basename(vp))[0]    if glob.glob(f'{CLIPS_DIR}/{vname}_clip*.mp4'):        print(f"[{vi+1}/{len(videos)}] {os.path.basename(vp)} \u2014 already processed, skipping")        continue    print(f"[{vi+1}/{len(videos)}] Processing {os.path.basename(vp)}...")    try:        scenes = detect(vp, ContentDetector(threshold=27.0))        cap = cv2.VideoCapture(vp)        fps = cap.get(cv2.CAP_PROP_FPS) or 30        clip_n = 0        for s, e in scenes:            dur = (e.get_frames() - s.get_frames()) / fps            if 2 <= dur <= 15:                cp = f'{CLIPS_DIR}/{vname}_clip{clip_n:04d}.mp4'                start_sec = s.get_frames() / fps                subprocess.run(                    ['ffmpeg', '-y', '-ss', str(start_sec), '-i', vp,                     '-t', str(dur), '-c:v', 'libx264', '-preset', 'fast',                     '-crf', '23', '-an', cp],                    capture_output=True                )                if os.path.exists(cp) and os.path.getsize(cp) > 10000:                    clip_n += 1                else:                    if os.path.exists(cp):                        os.remove(cp)        cap.release()        print(f"  \u2192 {clip_n} clips extracted")    except Exception as e:        print(f"  \u26a0\ufe0f Failed: {e}")total_clips = len(glob.glob(f'{CLIPS_DIR}/*.mp4'))print(f"\nLacrosse clips total: {total_clips}")

In [ ]:
# Cell 1G: Clip count summarybb_clips = len(glob.glob(f'{CLIPS_BASE}/basketball/*.mp4'))fb_clips = len(glob.glob(f'{CLIPS_BASE}/football/*.mp4'))lax_clips = len(glob.glob(f'{CLIPS_BASE}/lacrosse/*.mp4'))total = bb_clips + fb_clips + lax_clipsbb_vids = len(glob.glob(f'{RAW_VIDEO_BASE}/basketball/*'))fb_vids = len(glob.glob(f'{RAW_VIDEO_BASE}/football/*'))lax_vids = len(glob.glob(f'{RAW_VIDEO_BASE}/lacrosse/*'))print("=" * 50)print("\u2705 HARVESTING COMPLETE")print(f"Basketball videos downloaded: {bb_vids}")print(f"Basketball clips created: {bb_clips} \u2014 {'\u2705 50+' if bb_clips >= 50 else '\u26a0\ufe0f need 50+'}")print(f"Football videos downloaded: {fb_vids}")print(f"Football clips created: {fb_clips} \u2014 {'\u2705 50+' if fb_clips >= 50 else '\u26a0\ufe0f need 50+'}")print(f"Lacrosse videos downloaded: {lax_vids}")print(f"Lacrosse clips created: {lax_clips} \u2014 {'\u2705 50+' if lax_clips >= 50 else '\u26a0\ufe0f need 50+'}")print(f"Total clips: {total}")print("=" * 50)for sport, count in [('basketball', bb_clips), ('football', fb_clips), ('lacrosse', lax_clips)]:    if count < 50:        print(f"\033[93m\u26a0\ufe0f WARNING: {sport} only has {count} clips (need 50+)\033[0m")        print(f"   Try adding more search queries or specific video URLs to Cell 1{'A' if sport=='basketball' else 'C' if sport=='football' else 'E'}")

## SECTION 2: CLAUDE AUTO-LABELINGUse Claude Vision (Haiku) to classify each clip by play type.Extracts 3 frames per clip, sends to Claude, gets back a label.**Cost:** ~$0.003/clip with Haiku. 650 max API calls = ~$1.95 max.

In [ ]:
# Cell 2A: Label basketball clipsimport anthropicSPORT = 'basketball'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'# Guard: check clips existclip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert len(clip_files) > 0, f"\033[91m\u274c No clips in {CLIPS_DIR} \u2014 run Section 1 first\033[0m"# Load existing labels from Drivelabels = {}if os.path.exists(LABELS_FILE):    with open(LABELS_FILE) as f:        labels = json.load(f)    print(f"Loaded {len(labels)} existing labels from Drive")unlabeled = [c for c in clip_files if os.path.basename(c) not in labels]to_label = unlabeled[:MAX_CLIPS_TO_LABEL]print(f"Total clips: {len(clip_files)}, Already labeled: {len(labels)}, To label: {len(to_label)}")if not to_label:    print("\u2705 All clips already labeled!")else:    assert ANTHROPIC_API_KEY, "\033[91m\u274c Set ANTHROPIC_API_KEY in Colab Secrets\033[0m"    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)    types_list = ', '.join(PLAY_TYPES[SPORT])    cost = 0.0    api_calls = 0    for i, cp in enumerate(to_label):        # Hard stop on API calls        if api_calls >= MAX_API_CALLS_TOTAL:            print(f"\n\u26d4 Hard stop: {api_calls} API calls reached")            print(f"Estimated cost: ${api_calls * COST_PER_CALL_ESTIMATE:.2f}")            break        cn = os.path.basename(cp)        # Extract 3 frames        cap = cv2.VideoCapture(cp)        tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))        frame_b64s = []        for frac in [0.1, 0.5, 0.9]:            cap.set(cv2.CAP_PROP_POS_FRAMES, int(tf * frac))            ret, frame = cap.read()            if ret:                frame = cv2.resize(frame, (512, 288))                _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])                frame_b64s.append(base64.b64encode(buf).decode())        cap.release()        if len(frame_b64s) < 2:            labels[cn] = 'other'            continue        # Build message content        content = []        for b64 in frame_b64s:            content.append({"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}})        content.append({"type": "text", "text": f"This is a {SPORT} clip. Classify as one of: {types_list}. Reply ONLY the label, nothing else."})        try:            resp = client.messages.create(                model="claude-haiku-4-5-20251001",                max_tokens=20,                messages=[{"role": "user", "content": content}]            )            label = resp.content[0].text.strip().lower().replace(' ', '_')            if label not in PLAY_TYPES[SPORT]:                label = 'other'            labels[cn] = label            cost += (resp.usage.input_tokens * 0.25 + resp.usage.output_tokens * 1.25) / 1e6            api_calls += 1        except Exception as e:            labels[cn] = 'other'            api_calls += 1        # Progress every 10 clips        if (i + 1) % 10 == 0:            print(f"  [{i+1}/{len(to_label)}] labeled \u2014 cost so far: ${cost:.4f} \u2014 API calls: {api_calls}")            # Save checkpoint to Drive            with open(LABELS_FILE, 'w') as f:                json.dump(labels, f, indent=2)    # Final save    with open(LABELS_FILE, 'w') as f:        json.dump(labels, f, indent=2)    print(f"\n\u2705 Basketball labeling done! Cost: ${cost:.4f}, API calls: {api_calls}")# Print distributionif labels:    print(f"\nLabel distribution:")    for label, count in Counter(labels.values()).most_common():        print(f"  {label}: {count}")

In [ ]:
# Cell 2B: Label football clipsimport anthropicSPORT = 'football'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'clip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert len(clip_files) > 0, f"\033[91m\u274c No clips in {CLIPS_DIR} \u2014 run Section 1 first\033[0m"labels = {}if os.path.exists(LABELS_FILE):    with open(LABELS_FILE) as f:        labels = json.load(f)    print(f"Loaded {len(labels)} existing labels from Drive")unlabeled = [c for c in clip_files if os.path.basename(c) not in labels]to_label = unlabeled[:MAX_CLIPS_TO_LABEL]print(f"Total clips: {len(clip_files)}, Already labeled: {len(labels)}, To label: {len(to_label)}")if not to_label:    print("\u2705 All clips already labeled!")else:    assert ANTHROPIC_API_KEY, "\033[91m\u274c Set ANTHROPIC_API_KEY\033[0m"    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)    types_list = ', '.join(PLAY_TYPES[SPORT])    cost = 0.0    api_calls = 0    for i, cp in enumerate(to_label):        if api_calls >= MAX_API_CALLS_TOTAL:            print(f"\n\u26d4 Hard stop: {api_calls} API calls")            break        cn = os.path.basename(cp)        cap = cv2.VideoCapture(cp)        tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))        frame_b64s = []        for frac in [0.1, 0.5, 0.9]:            cap.set(cv2.CAP_PROP_POS_FRAMES, int(tf * frac))            ret, frame = cap.read()            if ret:                frame = cv2.resize(frame, (512, 288))                _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])                frame_b64s.append(base64.b64encode(buf).decode())        cap.release()        if len(frame_b64s) < 2:            labels[cn] = 'other'            continue        content = [{"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b}} for b in frame_b64s]        content.append({"type": "text", "text": f"This is a {SPORT} clip. Classify as one of: {types_list}. Reply ONLY the label."})        try:            resp = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=20, messages=[{"role": "user", "content": content}])            label = resp.content[0].text.strip().lower().replace(' ', '_')            if label not in PLAY_TYPES[SPORT]:                label = 'other'            labels[cn] = label            cost += (resp.usage.input_tokens * 0.25 + resp.usage.output_tokens * 1.25) / 1e6            api_calls += 1        except Exception:            labels[cn] = 'other'            api_calls += 1        if (i + 1) % 10 == 0:            print(f"  [{i+1}/{len(to_label)}] labeled \u2014 cost: ${cost:.4f} \u2014 calls: {api_calls}")            with open(LABELS_FILE, 'w') as f:                json.dump(labels, f, indent=2)    with open(LABELS_FILE, 'w') as f:        json.dump(labels, f, indent=2)    print(f"\n\u2705 Football labeling done! Cost: ${cost:.4f}, calls: {api_calls}")if labels:    print(f"\nLabel distribution:")    for label, count in Counter(labels.values()).most_common():        print(f"  {label}: {count}")

In [ ]:
# Cell 2C: Label lacrosse clipsimport anthropicSPORT = 'lacrosse'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'clip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert len(clip_files) > 0, f"\033[91m\u274c No clips in {CLIPS_DIR} \u2014 run Section 1 first\033[0m"labels = {}if os.path.exists(LABELS_FILE):    with open(LABELS_FILE) as f:        labels = json.load(f)    print(f"Loaded {len(labels)} existing labels from Drive")unlabeled = [c for c in clip_files if os.path.basename(c) not in labels]to_label = unlabeled[:MAX_CLIPS_TO_LABEL]print(f"Total clips: {len(clip_files)}, Already labeled: {len(labels)}, To label: {len(to_label)}")if not to_label:    print("\u2705 All clips already labeled!")else:    assert ANTHROPIC_API_KEY, "\033[91m\u274c Set ANTHROPIC_API_KEY\033[0m"    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)    types_list = ', '.join(PLAY_TYPES[SPORT])    cost = 0.0    api_calls = 0    for i, cp in enumerate(to_label):        if api_calls >= MAX_API_CALLS_TOTAL:            print(f"\n\u26d4 Hard stop: {api_calls} API calls")            break        cn = os.path.basename(cp)        cap = cv2.VideoCapture(cp)        tf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))        frame_b64s = []        for frac in [0.1, 0.5, 0.9]:            cap.set(cv2.CAP_PROP_POS_FRAMES, int(tf * frac))            ret, frame = cap.read()            if ret:                frame = cv2.resize(frame, (512, 288))                _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])                frame_b64s.append(base64.b64encode(buf).decode())        cap.release()        if len(frame_b64s) < 2:            labels[cn] = 'other'            continue        content = [{"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b}} for b in frame_b64s]        content.append({"type": "text", "text": f"This is a {SPORT} clip. Classify as one of: {types_list}. Reply ONLY the label."})        try:            resp = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=20, messages=[{"role": "user", "content": content}])            label = resp.content[0].text.strip().lower().replace(' ', '_')            if label not in PLAY_TYPES[SPORT]:                label = 'other'            labels[cn] = label            cost += (resp.usage.input_tokens * 0.25 + resp.usage.output_tokens * 1.25) / 1e6            api_calls += 1        except Exception:            labels[cn] = 'other'            api_calls += 1        if (i + 1) % 10 == 0:            print(f"  [{i+1}/{len(to_label)}] labeled \u2014 cost: ${cost:.4f} \u2014 calls: {api_calls}")            with open(LABELS_FILE, 'w') as f:                json.dump(labels, f, indent=2)    with open(LABELS_FILE, 'w') as f:        json.dump(labels, f, indent=2)    print(f"\n\u2705 Lacrosse labeling done! Cost: ${cost:.4f}, calls: {api_calls}")if labels:    print(f"\nLabel distribution:")    for label, count in Counter(labels.values()).most_common():        print(f"  {label}: {count}")

In [ ]:
# Cell 2D: Label summarytotal_api_calls = 0total_cost = 0.0print("=" * 50)print("\u2705 LABELING COMPLETE")for sport in ['basketball', 'football', 'lacrosse']:    lf = f'{DRIVE_LABELS}/{sport}_labels.json'    clips_dir = f'{CLIPS_BASE}/{sport}'    total_clips = len(glob.glob(f'{clips_dir}/*.mp4'))    if os.path.exists(lf):        with open(lf) as f:            labels = json.load(f)        print(f"{sport.capitalize()} clips labeled: {len(labels)}/{total_clips}")    else:        print(f"{sport.capitalize()} clips labeled: 0/{total_clips}")# Estimate total cost from all label filesfor sport in ['basketball', 'football', 'lacrosse']:    lf = f'{DRIVE_LABELS}/{sport}_labels.json'    if os.path.exists(lf):        with open(lf) as f:            labels = json.load(f)        total_api_calls += len(labels)est_cost = total_api_calls * COST_PER_CALL_ESTIMATEprint(f"Total API calls made: ~{total_api_calls}")print(f"Estimated cost: ${est_cost:.2f}")print(f"Labels saved to Drive: \u2705 {DRIVE_LABELS}/")print("=" * 50)if est_cost > 15:    print("\033[93m\u26a0\ufe0f WARNING: Estimated cost exceeds $15\033[0m")

## SECTION 3: VIDEOMAE TRAININGFine-tune VideoMAE (MCG-NJU/videomae-base-finetuned-kinetics) for play type classification.One model per sport. Output: `videomae_{sport}_v5.zip`

In [ ]:
# Cell 3A: Train videomae_basketball_v5from transformers import VideoMAEForVideoClassification, VideoMAEConfigfrom torch.utils.data import Dataset, DataLoaderfrom torch.optim import AdamWfrom torch.optim.lr_scheduler import CosineAnnealingLRSPORT = 'basketball'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'MODEL_NAME = f'videomae_{SPORT}_v5'LOCAL_MODEL_DIR = f'/content/models/{MODEL_NAME}'CHECKPOINT_FILE = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}_checkpoint.pt'# === GUARD ===clip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert os.path.exists(LABELS_FILE), f"\033[91m\u274c No labels file \u2014 run Section 2 first\033[0m"with open(LABELS_FILE) as f:    all_labels = json.load(f)assert len(clip_files) >= 50, f"\033[91m\u274c Need 50+ clips, have {len(clip_files)} \u2014 run Section 1\033[0m"assert len(all_labels) >= 50, f"\033[91m\u274c Need 50+ labels, have {len(all_labels)} \u2014 run Section 2\033[0m"print(f"\u2705 {len(clip_files)} clips, {len(all_labels)} labels ready")# === DATASET ===class ClipDataset(Dataset):    def __init__(self, clips_dir, labels_dict, play_types):        self.label2id = {t: i for i, t in enumerate(play_types)}        self.samples = []        for cp in sorted(glob.glob(f'{clips_dir}/*.mp4')):            cn = os.path.basename(cp)            if cn in labels_dict and labels_dict[cn] in self.label2id:                self.samples.append((cp, self.label2id[labels_dict[cn]]))        print(f"  Dataset: {len(self.samples)} samples, {len(play_types)} classes")    def __len__(self):        return len(self.samples)    def __getitem__(self, idx):        path, label = self.samples[idx]        cap = cv2.VideoCapture(path)        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))        indices = np.linspace(0, max(total - 1, 0), 16, dtype=int)        frames = []        for fi in indices:            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)            ret, frame = cap.read()            if ret:                frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224)).astype(np.float32) / 255.0)            else:                frames.append(np.zeros((224, 224, 3), dtype=np.float32))        cap.release()        pixel_values = torch.tensor(np.transpose(np.stack(frames), (0, 3, 1, 2)))        return {'pixel_values': pixel_values, 'labels': torch.tensor(label)}ds = ClipDataset(CLIPS_DIR, all_labels, PLAY_TYPES[SPORT])assert len(ds) >= 10, f"\033[91m\u274c Need 10+ matched samples, have {len(ds)}\033[0m"ts = int(0.8 * len(ds))vs = len(ds) - tstrain_ds, val_ds = torch.utils.data.random_split(ds, [ts, vs])print(f"  Train: {ts}, Val: {vs}")# === MODEL ===label2id = ds.label2idid2label = {v: k for k, v in label2id.items()}config = VideoMAEConfig.from_pretrained(    'MCG-NJU/videomae-base-finetuned-kinetics',    num_labels=len(label2id), label2id=label2id, id2label=id2label)model = VideoMAEForVideoClassification.from_pretrained(    'MCG-NJU/videomae-base-finetuned-kinetics',    config=config, ignore_mismatched_sizes=True).cuda()# Resume from checkpoint if existsstart_epoch = 0best_acc = 0.0if os.path.exists(CHECKPOINT_FILE):    ckpt = torch.load(CHECKPOINT_FILE)    model.load_state_dict(ckpt['model_state'])    start_epoch = ckpt.get('epoch', 0)    best_acc = ckpt.get('val_acc', 0.0)    print(f"  Resumed from epoch {start_epoch}, best_acc={best_acc:.3f}")# === TRAIN ===optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)scheduler = CosineAnnealingLR(optimizer, T_max=20)train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)val_loader = DataLoader(val_ds, batch_size=4, num_workers=2)patience_counter = 0for epoch in range(start_epoch, 20):    # Train    model.train()    total_loss = 0    correct = 0    total = 0    for batch in train_loader:        pv = batch['pixel_values'].cuda()        lb = batch['labels'].cuda()        out = model(pixel_values=pv, labels=lb)        optimizer.zero_grad()        out.loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        optimizer.step()        total_loss += out.loss.item()        correct += (out.logits.argmax(-1) == lb).sum().item()        total += lb.size(0)    scheduler.step()    # Validate    model.eval()    val_correct = 0    val_total = 0    with torch.no_grad():        for batch in val_loader:            pv = batch['pixel_values'].cuda()            lb = batch['labels'].cuda()            val_correct += (model(pixel_values=pv).logits.argmax(-1) == lb).sum().item()            val_total += lb.size(0)    val_acc = val_correct / max(val_total, 1)    print(f"  Epoch {epoch+1}/20 \u2014 loss:{total_loss/len(train_loader):.4f} train_acc:{correct/max(total,1):.3f} val_acc:{val_acc:.3f}")    if val_acc > best_acc:        best_acc = val_acc        patience_counter = 0        os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)        model.save_pretrained(LOCAL_MODEL_DIR)        torch.save({'model_state': model.state_dict(), 'epoch': epoch + 1,                     'val_acc': val_acc, 'label2id': label2id}, CHECKPOINT_FILE)        print(f"  [BEST] val_acc={val_acc:.3f} \u2014 saved")    else:        patience_counter += 1        if patience_counter >= 5:            print(f"  Early stopping at epoch {epoch+1}")            breaktorch.cuda.empty_cache()gc.collect()print(f"\n\u2705 Training complete! Best val_acc={best_acc:.3f}")

In [ ]:
# Cell 3B: Save videomae_basketball_v5.zip to DriveMODEL_NAME = 'videomae_basketball_v5'LOCAL_MODEL_DIR = f'/content/models/{MODEL_NAME}'ZIP_PATH = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.zip'# Guardassert os.path.exists(LOCAL_MODEL_DIR), f"\033[91m\u274c No trained model at {LOCAL_MODEL_DIR} \u2014 run Cell 3A first\033[0m"assert len(glob.glob(f'{LOCAL_MODEL_DIR}/*')) > 0, f"\033[91m\u274c Model directory is empty\033[0m"# Zip and save to Driveimport zipfilewith zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:    for root, dirs, files in os.walk(LOCAL_MODEL_DIR):        for fn in files:            fp = os.path.join(root, fn)            arcname = os.path.relpath(fp, LOCAL_MODEL_DIR)            zf.write(fp, arcname)sz = os.path.getsize(ZIP_PATH) / 1024 / 1024print("=" * 50)print(f"\u2705 {MODEL_NAME}.zip SAVED TO DRIVE")print(f"Location: {DRIVE_SAVE_DIR}/")print(f"File size: {sz:.1f}MB")print("=" * 50)

In [ ]:
# Cell 3C: Train videomae_football_v5from transformers import VideoMAEForVideoClassification, VideoMAEConfigfrom torch.utils.data import Dataset, DataLoaderfrom torch.optim import AdamWfrom torch.optim.lr_scheduler import CosineAnnealingLRSPORT = 'football'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'MODEL_NAME = f'videomae_{SPORT}_v5'LOCAL_MODEL_DIR = f'/content/models/{MODEL_NAME}'CHECKPOINT_FILE = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}_checkpoint.pt'# === GUARD ===clip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert os.path.exists(LABELS_FILE), f"\033[91m\u274c No labels file \u2014 run Section 2 first\033[0m"with open(LABELS_FILE) as f:    all_labels = json.load(f)assert len(clip_files) >= 50, f"\033[91m\u274c Need 50+ clips, have {len(clip_files)} \u2014 run Section 1\033[0m"assert len(all_labels) >= 50, f"\033[91m\u274c Need 50+ labels, have {len(all_labels)} \u2014 run Section 2\033[0m"print(f"\u2705 {len(clip_files)} clips, {len(all_labels)} labels ready")# === DATASET ===class ClipDataset(Dataset):    def __init__(self, clips_dir, labels_dict, play_types):        self.label2id = {t: i for i, t in enumerate(play_types)}        self.samples = []        for cp in sorted(glob.glob(f'{clips_dir}/*.mp4')):            cn = os.path.basename(cp)            if cn in labels_dict and labels_dict[cn] in self.label2id:                self.samples.append((cp, self.label2id[labels_dict[cn]]))        print(f"  Dataset: {len(self.samples)} samples, {len(play_types)} classes")    def __len__(self):        return len(self.samples)    def __getitem__(self, idx):        path, label = self.samples[idx]        cap = cv2.VideoCapture(path)        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))        indices = np.linspace(0, max(total - 1, 0), 16, dtype=int)        frames = []        for fi in indices:            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)            ret, frame = cap.read()            if ret:                frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224)).astype(np.float32) / 255.0)            else:                frames.append(np.zeros((224, 224, 3), dtype=np.float32))        cap.release()        pixel_values = torch.tensor(np.transpose(np.stack(frames), (0, 3, 1, 2)))        return {'pixel_values': pixel_values, 'labels': torch.tensor(label)}ds = ClipDataset(CLIPS_DIR, all_labels, PLAY_TYPES[SPORT])assert len(ds) >= 10, f"\033[91m\u274c Need 10+ matched samples, have {len(ds)}\033[0m"ts = int(0.8 * len(ds))vs = len(ds) - tstrain_ds, val_ds = torch.utils.data.random_split(ds, [ts, vs])print(f"  Train: {ts}, Val: {vs}")# === MODEL ===label2id = ds.label2idid2label = {v: k for k, v in label2id.items()}config = VideoMAEConfig.from_pretrained(    'MCG-NJU/videomae-base-finetuned-kinetics',    num_labels=len(label2id), label2id=label2id, id2label=id2label)model = VideoMAEForVideoClassification.from_pretrained(    'MCG-NJU/videomae-base-finetuned-kinetics',    config=config, ignore_mismatched_sizes=True).cuda()# Resume from checkpoint if existsstart_epoch = 0best_acc = 0.0if os.path.exists(CHECKPOINT_FILE):    ckpt = torch.load(CHECKPOINT_FILE)    model.load_state_dict(ckpt['model_state'])    start_epoch = ckpt.get('epoch', 0)    best_acc = ckpt.get('val_acc', 0.0)    print(f"  Resumed from epoch {start_epoch}, best_acc={best_acc:.3f}")# === TRAIN ===optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)scheduler = CosineAnnealingLR(optimizer, T_max=20)train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)val_loader = DataLoader(val_ds, batch_size=4, num_workers=2)patience_counter = 0for epoch in range(start_epoch, 20):    # Train    model.train()    total_loss = 0    correct = 0    total = 0    for batch in train_loader:        pv = batch['pixel_values'].cuda()        lb = batch['labels'].cuda()        out = model(pixel_values=pv, labels=lb)        optimizer.zero_grad()        out.loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        optimizer.step()        total_loss += out.loss.item()        correct += (out.logits.argmax(-1) == lb).sum().item()        total += lb.size(0)    scheduler.step()    # Validate    model.eval()    val_correct = 0    val_total = 0    with torch.no_grad():        for batch in val_loader:            pv = batch['pixel_values'].cuda()            lb = batch['labels'].cuda()            val_correct += (model(pixel_values=pv).logits.argmax(-1) == lb).sum().item()            val_total += lb.size(0)    val_acc = val_correct / max(val_total, 1)    print(f"  Epoch {epoch+1}/20 \u2014 loss:{total_loss/len(train_loader):.4f} train_acc:{correct/max(total,1):.3f} val_acc:{val_acc:.3f}")    if val_acc > best_acc:        best_acc = val_acc        patience_counter = 0        os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)        model.save_pretrained(LOCAL_MODEL_DIR)        torch.save({'model_state': model.state_dict(), 'epoch': epoch + 1,                     'val_acc': val_acc, 'label2id': label2id}, CHECKPOINT_FILE)        print(f"  [BEST] val_acc={val_acc:.3f} \u2014 saved")    else:        patience_counter += 1        if patience_counter >= 5:            print(f"  Early stopping at epoch {epoch+1}")            breaktorch.cuda.empty_cache()gc.collect()print(f"\n\u2705 Training complete! Best val_acc={best_acc:.3f}")

In [ ]:
# Cell 3D: Save videomae_football_v5.zip to DriveMODEL_NAME = 'videomae_football_v5'LOCAL_MODEL_DIR = f'/content/models/{MODEL_NAME}'ZIP_PATH = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.zip'# Guardassert os.path.exists(LOCAL_MODEL_DIR), f"\033[91m\u274c No trained model at {LOCAL_MODEL_DIR} \u2014 run Cell 3C first\033[0m"assert len(glob.glob(f'{LOCAL_MODEL_DIR}/*')) > 0, f"\033[91m\u274c Model directory is empty\033[0m"# Zip and save to Driveimport zipfilewith zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:    for root, dirs, files in os.walk(LOCAL_MODEL_DIR):        for fn in files:            fp = os.path.join(root, fn)            arcname = os.path.relpath(fp, LOCAL_MODEL_DIR)            zf.write(fp, arcname)sz = os.path.getsize(ZIP_PATH) / 1024 / 1024print("=" * 50)print(f"\u2705 {MODEL_NAME}.zip SAVED TO DRIVE")print(f"Location: {DRIVE_SAVE_DIR}/")print(f"File size: {sz:.1f}MB")print("=" * 50)

In [ ]:
# Cell 3E: Train videomae_lacrosse_v5from transformers import VideoMAEForVideoClassification, VideoMAEConfigfrom torch.utils.data import Dataset, DataLoaderfrom torch.optim import AdamWfrom torch.optim.lr_scheduler import CosineAnnealingLRSPORT = 'lacrosse'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'MODEL_NAME = f'videomae_{SPORT}_v5'LOCAL_MODEL_DIR = f'/content/models/{MODEL_NAME}'CHECKPOINT_FILE = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}_checkpoint.pt'# === GUARD ===clip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert os.path.exists(LABELS_FILE), f"\033[91m\u274c No labels file \u2014 run Section 2 first\033[0m"with open(LABELS_FILE) as f:    all_labels = json.load(f)assert len(clip_files) >= 50, f"\033[91m\u274c Need 50+ clips, have {len(clip_files)} \u2014 run Section 1\033[0m"assert len(all_labels) >= 50, f"\033[91m\u274c Need 50+ labels, have {len(all_labels)} \u2014 run Section 2\033[0m"print(f"\u2705 {len(clip_files)} clips, {len(all_labels)} labels ready")# === DATASET ===class ClipDataset(Dataset):    def __init__(self, clips_dir, labels_dict, play_types):        self.label2id = {t: i for i, t in enumerate(play_types)}        self.samples = []        for cp in sorted(glob.glob(f'{clips_dir}/*.mp4')):            cn = os.path.basename(cp)            if cn in labels_dict and labels_dict[cn] in self.label2id:                self.samples.append((cp, self.label2id[labels_dict[cn]]))        print(f"  Dataset: {len(self.samples)} samples, {len(play_types)} classes")    def __len__(self):        return len(self.samples)    def __getitem__(self, idx):        path, label = self.samples[idx]        cap = cv2.VideoCapture(path)        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))        indices = np.linspace(0, max(total - 1, 0), 16, dtype=int)        frames = []        for fi in indices:            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)            ret, frame = cap.read()            if ret:                frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224)).astype(np.float32) / 255.0)            else:                frames.append(np.zeros((224, 224, 3), dtype=np.float32))        cap.release()        pixel_values = torch.tensor(np.transpose(np.stack(frames), (0, 3, 1, 2)))        return {'pixel_values': pixel_values, 'labels': torch.tensor(label)}ds = ClipDataset(CLIPS_DIR, all_labels, PLAY_TYPES[SPORT])assert len(ds) >= 10, f"\033[91m\u274c Need 10+ matched samples, have {len(ds)}\033[0m"ts = int(0.8 * len(ds))vs = len(ds) - tstrain_ds, val_ds = torch.utils.data.random_split(ds, [ts, vs])print(f"  Train: {ts}, Val: {vs}")# === MODEL ===label2id = ds.label2idid2label = {v: k for k, v in label2id.items()}config = VideoMAEConfig.from_pretrained(    'MCG-NJU/videomae-base-finetuned-kinetics',    num_labels=len(label2id), label2id=label2id, id2label=id2label)model = VideoMAEForVideoClassification.from_pretrained(    'MCG-NJU/videomae-base-finetuned-kinetics',    config=config, ignore_mismatched_sizes=True).cuda()# Resume from checkpoint if existsstart_epoch = 0best_acc = 0.0if os.path.exists(CHECKPOINT_FILE):    ckpt = torch.load(CHECKPOINT_FILE)    model.load_state_dict(ckpt['model_state'])    start_epoch = ckpt.get('epoch', 0)    best_acc = ckpt.get('val_acc', 0.0)    print(f"  Resumed from epoch {start_epoch}, best_acc={best_acc:.3f}")# === TRAIN ===optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)scheduler = CosineAnnealingLR(optimizer, T_max=20)train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)val_loader = DataLoader(val_ds, batch_size=4, num_workers=2)patience_counter = 0for epoch in range(start_epoch, 20):    # Train    model.train()    total_loss = 0    correct = 0    total = 0    for batch in train_loader:        pv = batch['pixel_values'].cuda()        lb = batch['labels'].cuda()        out = model(pixel_values=pv, labels=lb)        optimizer.zero_grad()        out.loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        optimizer.step()        total_loss += out.loss.item()        correct += (out.logits.argmax(-1) == lb).sum().item()        total += lb.size(0)    scheduler.step()    # Validate    model.eval()    val_correct = 0    val_total = 0    with torch.no_grad():        for batch in val_loader:            pv = batch['pixel_values'].cuda()            lb = batch['labels'].cuda()            val_correct += (model(pixel_values=pv).logits.argmax(-1) == lb).sum().item()            val_total += lb.size(0)    val_acc = val_correct / max(val_total, 1)    print(f"  Epoch {epoch+1}/20 \u2014 loss:{total_loss/len(train_loader):.4f} train_acc:{correct/max(total,1):.3f} val_acc:{val_acc:.3f}")    if val_acc > best_acc:        best_acc = val_acc        patience_counter = 0        os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)        model.save_pretrained(LOCAL_MODEL_DIR)        torch.save({'model_state': model.state_dict(), 'epoch': epoch + 1,                     'val_acc': val_acc, 'label2id': label2id}, CHECKPOINT_FILE)        print(f"  [BEST] val_acc={val_acc:.3f} \u2014 saved")    else:        patience_counter += 1        if patience_counter >= 5:            print(f"  Early stopping at epoch {epoch+1}")            breaktorch.cuda.empty_cache()gc.collect()print(f"\n\u2705 Training complete! Best val_acc={best_acc:.3f}")

In [ ]:
# Cell 3F: Save videomae_lacrosse_v5.zip to DriveMODEL_NAME = 'videomae_lacrosse_v5'LOCAL_MODEL_DIR = f'/content/models/{MODEL_NAME}'ZIP_PATH = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.zip'# Guardassert os.path.exists(LOCAL_MODEL_DIR), f"\033[91m\u274c No trained model at {LOCAL_MODEL_DIR} \u2014 run Cell 3E first\033[0m"assert len(glob.glob(f'{LOCAL_MODEL_DIR}/*')) > 0, f"\033[91m\u274c Model directory is empty\033[0m"# Zip and save to Driveimport zipfilewith zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:    for root, dirs, files in os.walk(LOCAL_MODEL_DIR):        for fn in files:            fp = os.path.join(root, fn)            arcname = os.path.relpath(fp, LOCAL_MODEL_DIR)            zf.write(fp, arcname)sz = os.path.getsize(ZIP_PATH) / 1024 / 1024print("=" * 50)print(f"\u2705 {MODEL_NAME}.zip SAVED TO DRIVE")print(f"Location: {DRIVE_SAVE_DIR}/")print(f"File size: {sz:.1f}MB")print("=" * 50)

## SECTION 4: YOLO OUTCOME CLASSIFIERSTrain YOLOv8 classification models to classify play outcomes from single frames.One model per sport. Output: `outcome_classifier_{sport}_v5.pt`

In [ ]:
# Cell 4A: Train outcome_classifier_basketball_v5from ultralytics import YOLOSPORT = 'basketball'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'MODEL_NAME = f'outcome_classifier_{SPORT}_v5'CLS_DIR = f'/content/yolo_cls/{SPORT}'# === GUARD ===clip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert os.path.exists(LABELS_FILE), f"\033[91m\u274c No labels \u2014 run Section 2 first\033[0m"with open(LABELS_FILE) as f:    all_labels = json.load(f)assert len(clip_files) >= 50, f"\033[91m\u274c Need 50+ clips, have {len(clip_files)} \u2014 run Section 1\033[0m"assert len(all_labels) >= 50, f"\033[91m\u274c Need 50+ labels, have {len(all_labels)} \u2014 run Section 2\033[0m"print(f"\u2705 {len(clip_files)} clips, {len(all_labels)} labels ready")# === BUILD CLASSIFICATION DATASET ===# Extract middle frame from each clip, organize into class foldersfor cn, label in all_labels.items():    cp = f'{CLIPS_DIR}/{cn}'    if not os.path.exists(cp):        continue    cap = cv2.VideoCapture(cp)    mid_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2    cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)    ret, frame = cap.read()    cap.release()    if not ret:        continue    split = 'train' if hash(cn) % 5 != 0 else 'val'    out_dir = f'{CLS_DIR}/{split}/{label}'    os.makedirs(out_dir, exist_ok=True)    cv2.imwrite(f'{out_dir}/{cn.replace(".mp4", ".jpg")}', cv2.resize(frame, (640, 640)))train_count = len(glob.glob(f'{CLS_DIR}/train/**/*.jpg', recursive=True))val_count = len(glob.glob(f'{CLS_DIR}/val/**/*.jpg', recursive=True))print(f"Classification dataset: {train_count} train, {val_count} val")assert train_count >= 20, f"\033[91m\u274c Need 20+ training images, have {train_count}\033[0m"# === TRAIN ===# Resume from Drive checkpoint if existsresume_path = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/last.pt'if os.path.exists(resume_path) and os.path.getsize(resume_path) > 1024 * 1024:    print(f"Resuming from Drive checkpoint...")    model = YOLO(resume_path)    model.train(resume=True)else:    model = YOLO('yolov8m-cls.pt')    model.train(        data=CLS_DIR,        epochs=80,        imgsz=640,        batch=DEFAULT_BATCH,        name=MODEL_NAME,        device=0,        patience=15,        save_period=5,        amp=True,        cache=True,    )# Save checkpoint to Driveckpt_dir = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}'os.makedirs(ckpt_dir, exist_ok=True)for wt in ['best.pt', 'last.pt']:    paths = sorted(glob.glob(f'runs/classify/{MODEL_NAME}*/weights/{wt}'))    if paths:        shutil.copy2(paths[-1], f'{ckpt_dir}/{wt}')torch.cuda.empty_cache()gc.collect()print(f"\n\u2705 {MODEL_NAME} training complete!")

In [ ]:
# Cell 4B: Save outcome_classifier_basketball_v5.pt to DriveMODEL_NAME = 'outcome_classifier_basketball_v5'# Find best.ptpaths = sorted(glob.glob(f'runs/classify/{MODEL_NAME}*/weights/best.pt'))if not paths:    # Try Drive checkpoint    paths = [f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/best.pt']assert paths and os.path.exists(paths[-1] if paths else ''), f"\033[91m\u274c No trained model \u2014 run Cell 4A first\033[0m"src = paths[-1]dst = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.pt'shutil.copy2(src, dst)sz = os.path.getsize(dst) / 1024 / 1024print("=" * 50)print(f"\u2705 {MODEL_NAME}.pt SAVED TO DRIVE")print(f"Location: {DRIVE_SAVE_DIR}/")print(f"File size: {sz:.1f}MB")print("=" * 50)

In [ ]:
# Cell 4C: Train outcome_classifier_football_v5from ultralytics import YOLOSPORT = 'football'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'MODEL_NAME = f'outcome_classifier_{SPORT}_v5'CLS_DIR = f'/content/yolo_cls/{SPORT}'# === GUARD ===clip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert os.path.exists(LABELS_FILE), f"\033[91m\u274c No labels \u2014 run Section 2 first\033[0m"with open(LABELS_FILE) as f:    all_labels = json.load(f)assert len(clip_files) >= 50, f"\033[91m\u274c Need 50+ clips, have {len(clip_files)} \u2014 run Section 1\033[0m"assert len(all_labels) >= 50, f"\033[91m\u274c Need 50+ labels, have {len(all_labels)} \u2014 run Section 2\033[0m"print(f"\u2705 {len(clip_files)} clips, {len(all_labels)} labels ready")# === BUILD CLASSIFICATION DATASET ===# Extract middle frame from each clip, organize into class foldersfor cn, label in all_labels.items():    cp = f'{CLIPS_DIR}/{cn}'    if not os.path.exists(cp):        continue    cap = cv2.VideoCapture(cp)    mid_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2    cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)    ret, frame = cap.read()    cap.release()    if not ret:        continue    split = 'train' if hash(cn) % 5 != 0 else 'val'    out_dir = f'{CLS_DIR}/{split}/{label}'    os.makedirs(out_dir, exist_ok=True)    cv2.imwrite(f'{out_dir}/{cn.replace(".mp4", ".jpg")}', cv2.resize(frame, (640, 640)))train_count = len(glob.glob(f'{CLS_DIR}/train/**/*.jpg', recursive=True))val_count = len(glob.glob(f'{CLS_DIR}/val/**/*.jpg', recursive=True))print(f"Classification dataset: {train_count} train, {val_count} val")assert train_count >= 20, f"\033[91m\u274c Need 20+ training images, have {train_count}\033[0m"# === TRAIN ===# Resume from Drive checkpoint if existsresume_path = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/last.pt'if os.path.exists(resume_path) and os.path.getsize(resume_path) > 1024 * 1024:    print(f"Resuming from Drive checkpoint...")    model = YOLO(resume_path)    model.train(resume=True)else:    model = YOLO('yolov8m-cls.pt')    model.train(        data=CLS_DIR,        epochs=80,        imgsz=640,        batch=DEFAULT_BATCH,        name=MODEL_NAME,        device=0,        patience=15,        save_period=5,        amp=True,        cache=True,    )# Save checkpoint to Driveckpt_dir = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}'os.makedirs(ckpt_dir, exist_ok=True)for wt in ['best.pt', 'last.pt']:    paths = sorted(glob.glob(f'runs/classify/{MODEL_NAME}*/weights/{wt}'))    if paths:        shutil.copy2(paths[-1], f'{ckpt_dir}/{wt}')torch.cuda.empty_cache()gc.collect()print(f"\n\u2705 {MODEL_NAME} training complete!")

In [ ]:
# Cell 4D: Save outcome_classifier_football_v5.pt to DriveMODEL_NAME = 'outcome_classifier_football_v5'# Find best.ptpaths = sorted(glob.glob(f'runs/classify/{MODEL_NAME}*/weights/best.pt'))if not paths:    # Try Drive checkpoint    paths = [f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/best.pt']assert paths and os.path.exists(paths[-1] if paths else ''), f"\033[91m\u274c No trained model \u2014 run Cell 4C first\033[0m"src = paths[-1]dst = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.pt'shutil.copy2(src, dst)sz = os.path.getsize(dst) / 1024 / 1024print("=" * 50)print(f"\u2705 {MODEL_NAME}.pt SAVED TO DRIVE")print(f"Location: {DRIVE_SAVE_DIR}/")print(f"File size: {sz:.1f}MB")print("=" * 50)

In [ ]:
# Cell 4E: Train outcome_classifier_lacrosse_v5from ultralytics import YOLOSPORT = 'lacrosse'CLIPS_DIR = f'{CLIPS_BASE}/{SPORT}'LABELS_FILE = f'{DRIVE_LABELS}/{SPORT}_labels.json'MODEL_NAME = f'outcome_classifier_{SPORT}_v5'CLS_DIR = f'/content/yolo_cls/{SPORT}'# === GUARD ===clip_files = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))assert os.path.exists(LABELS_FILE), f"\033[91m\u274c No labels \u2014 run Section 2 first\033[0m"with open(LABELS_FILE) as f:    all_labels = json.load(f)assert len(clip_files) >= 50, f"\033[91m\u274c Need 50+ clips, have {len(clip_files)} \u2014 run Section 1\033[0m"assert len(all_labels) >= 50, f"\033[91m\u274c Need 50+ labels, have {len(all_labels)} \u2014 run Section 2\033[0m"print(f"\u2705 {len(clip_files)} clips, {len(all_labels)} labels ready")# === BUILD CLASSIFICATION DATASET ===# Extract middle frame from each clip, organize into class foldersfor cn, label in all_labels.items():    cp = f'{CLIPS_DIR}/{cn}'    if not os.path.exists(cp):        continue    cap = cv2.VideoCapture(cp)    mid_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2    cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)    ret, frame = cap.read()    cap.release()    if not ret:        continue    split = 'train' if hash(cn) % 5 != 0 else 'val'    out_dir = f'{CLS_DIR}/{split}/{label}'    os.makedirs(out_dir, exist_ok=True)    cv2.imwrite(f'{out_dir}/{cn.replace(".mp4", ".jpg")}', cv2.resize(frame, (640, 640)))train_count = len(glob.glob(f'{CLS_DIR}/train/**/*.jpg', recursive=True))val_count = len(glob.glob(f'{CLS_DIR}/val/**/*.jpg', recursive=True))print(f"Classification dataset: {train_count} train, {val_count} val")assert train_count >= 20, f"\033[91m\u274c Need 20+ training images, have {train_count}\033[0m"# === TRAIN ===# Resume from Drive checkpoint if existsresume_path = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/last.pt'if os.path.exists(resume_path) and os.path.getsize(resume_path) > 1024 * 1024:    print(f"Resuming from Drive checkpoint...")    model = YOLO(resume_path)    model.train(resume=True)else:    model = YOLO('yolov8m-cls.pt')    model.train(        data=CLS_DIR,        epochs=80,        imgsz=640,        batch=DEFAULT_BATCH,        name=MODEL_NAME,        device=0,        patience=15,        save_period=5,        amp=True,        cache=True,    )# Save checkpoint to Driveckpt_dir = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}'os.makedirs(ckpt_dir, exist_ok=True)for wt in ['best.pt', 'last.pt']:    paths = sorted(glob.glob(f'runs/classify/{MODEL_NAME}*/weights/{wt}'))    if paths:        shutil.copy2(paths[-1], f'{ckpt_dir}/{wt}')torch.cuda.empty_cache()gc.collect()print(f"\n\u2705 {MODEL_NAME} training complete!")

In [ ]:
# Cell 4F: Save outcome_classifier_lacrosse_v5.pt to DriveMODEL_NAME = 'outcome_classifier_lacrosse_v5'# Find best.ptpaths = sorted(glob.glob(f'runs/classify/{MODEL_NAME}*/weights/best.pt'))if not paths:    # Try Drive checkpoint    paths = [f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/best.pt']assert paths and os.path.exists(paths[-1] if paths else ''), f"\033[91m\u274c No trained model \u2014 run Cell 4E first\033[0m"src = paths[-1]dst = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.pt'shutil.copy2(src, dst)sz = os.path.getsize(dst) / 1024 / 1024print("=" * 50)print(f"\u2705 {MODEL_NAME}.pt SAVED TO DRIVE")print(f"Location: {DRIVE_SAVE_DIR}/")print(f"File size: {sz:.1f}MB")print("=" * 50)

## SECTION 5: JERSEY OCR UNIVERSALTrain a universal jersey number detector (digits 0-9).Combines Roboflow datasets + synthetic digit images.**Verified datasets:**- `footballplayertracking/jerseynumberdetectordigitdetector` v6 (primary)- `volleyai-actions/jersey-number-detection-s01j4` v2 (6,932 images)Output: `jersey_ocr_universal_v5.pt`

In [ ]:
# Cell 5A: Download Roboflow OCR datasets + prepare merged datafrom roboflow import Roboflowfrom PIL import Image, ImageDraw, ImageFontimport randomassert ROBOFLOW_API_KEY, "\033[91m\u274c Set ROBOFLOW_API_KEY in Colab Secrets\033[0m"rf = Roboflow(api_key=ROBOFLOW_API_KEY)# === Download datasets ===def safe_download(ws, proj, ver, name):    for v in [ver] + [i for i in range(1, 6) if i != ver]:        try:            ds = rf.workspace(ws).project(proj).version(v).download('yolov8')            count = len(glob.glob(f'{ds.location}/train/images/*'))            print(f"\u2705 {name} v{v}: {count} images")            return ds        except Exception as e:            print(f"  v{v} failed: {e}")    print(f"\033[91m\u274c {name}: all versions failed\033[0m")    return Noneprint("=== Downloading Jersey OCR Datasets ===")ds_jersey_1 = safe_download('footballplayertracking', 'jerseynumberdetectordigitdetector', 6, 'jersey_primary')ds_jersey_2 = safe_download('volleyai-actions', 'jersey-number-detection-s01j4', 2, 'jersey_volleyai')# === Remap classes to digits 0-9 ===def remap_to_digits(ds_dir):    for label_dir in glob.glob(f'{ds_dir}/*/labels'):        yaml_path = os.path.join(os.path.dirname(label_dir), 'data.yaml')        class_map = {}        if os.path.exists(yaml_path):            with open(yaml_path) as f:                d = yaml.safe_load(f)            for old_id, name in enumerate(d.get('names', [])):                clean = str(name).strip()                if clean.isdigit() and 0 <= int(clean) <= 9:                    class_map[old_id] = int(clean)        if not class_map:            continue        for lf in glob.glob(f'{label_dir}/*.txt'):            lines = []            with open(lf) as f:                for line in f:                    parts = line.strip().split()                    if len(parts) >= 5 and int(parts[0]) in class_map:                        parts[0] = str(class_map[int(parts[0])])                        lines.append(' '.join(parts))            if lines:                with open(lf, 'w') as f:                    f.write('\n'.join(lines) + '\n')for ds in [ds_jersey_1, ds_jersey_2]:    if ds:        remap_to_digits(ds.location)# === Generate 2000 synthetic digit images ===print("\nGenerating synthetic digit images...")SYNTH = '/content/synth_ocr'os.makedirs(f'{SYNTH}/images', exist_ok=True)os.makedirs(f'{SYNTH}/labels', exist_ok=True)COLORS = [(255,0,0),(0,0,255),(255,255,255),(0,128,0),(255,165,0),(128,0,128),(0,0,0)]for i in range(2000):    num = random.randint(0, 99)    digits = str(num)    w, h = 640, 640    bg = random.choice(COLORS)    img = Image.new('RGB', (w, h), bg)    draw = ImageDraw.Draw(img)    # Add noise    for _ in range(random.randint(50, 200)):        x1, y1 = random.randint(0, w-1), random.randint(0, h-1)        nc = tuple(max(0, min(255, c + random.randint(-30, 30))) for c in bg)        draw.rectangle([x1, y1, x1+random.randint(2,8), y1+random.randint(2,8)], fill=nc)    tc = random.choice([(255,255,255),(0,0,0),(255,255,0)])    fs = random.randint(60, 150)    try:        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', fs)    except:        font = ImageFont.load_default()    tw = len(digits) * (fs * 0.7)    sx = (w - tw) / 2 + random.randint(-50, 50)    yp = h / 2 - fs / 2 + random.randint(-80, 80)    label_lines = []    for j, d in enumerate(digits):        x = sx + j * (fs * 0.7)        draw.text((x, yp), d, fill=tc, font=font)        dw, dh = fs * 0.65, fs * 1.1        cx = max(0.01, min(0.99, (x + dw/2) / w))        cy = max(0.01, min(0.99, (yp + dh/2) / h))        bw = max(0.02, min(0.5, dw / w))        bh = max(0.02, min(0.5, dh / h))        label_lines.append(f'{int(d)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')    img.save(f'{SYNTH}/images/synth_{i:05d}.jpg')    with open(f'{SYNTH}/labels/synth_{i:05d}.txt', 'w') as f:        f.write('\n'.join(label_lines) + '\n')    if (i + 1) % 500 == 0:        print(f"  Generated {i+1}/2000 synthetic images")# === Merge all datasets ===print("\nMerging datasets...")MERGED = '/content/ocr_merged'for s in ['train', 'val']:    os.makedirs(f'{MERGED}/{s}/images', exist_ok=True)    os.makedirs(f'{MERGED}/{s}/labels', exist_ok=True)# Add Roboflow datasetsfor ds in [ds_jersey_1, ds_jersey_2]:    if not ds:        continue    loc = ds.location    dn = os.path.basename(loc)    for sp in ['train', 'valid', 'val', 'test']:        idir = f'{loc}/{sp}/images'        ldir = f'{loc}/{sp}/labels'        if not os.path.exists(idir):            continue        target_split = 'val' if sp in ['valid', 'val', 'test'] else 'train'        for img_path in glob.glob(f'{idir}/*'):            fn = f'{dn}_{os.path.basename(img_path)}'            shutil.copy2(img_path, f'{MERGED}/{target_split}/images/{fn}')            lbl = os.path.join(ldir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')            if os.path.exists(lbl):                shutil.copy2(lbl, f'{MERGED}/{target_split}/labels/{os.path.splitext(fn)[0]}.txt')# Add synthetic images (80% train, 20% val)synth_images = sorted(glob.glob(f'{SYNTH}/images/*.jpg'))split_point = int(len(synth_images) * 0.8)for i, ip in enumerate(synth_images):    fn = os.path.basename(ip)    s = 'train' if i < split_point else 'val'    shutil.copy2(ip, f'{MERGED}/{s}/images/{fn}')    shutil.copy2(f'{SYNTH}/labels/{fn.replace(".jpg", ".txt")}', f'{MERGED}/{s}/labels/{fn.replace(".jpg", ".txt")}')# Write data.yamldata_yaml = {    'path': MERGED,    'train': 'train/images',    'val': 'val/images',    'nc': 10,    'names': {i: str(i) for i in range(10)}}with open(f'{MERGED}/data.yaml', 'w') as f:    yaml.dump(data_yaml, f)tc = len(glob.glob(f'{MERGED}/train/images/*'))vc = len(glob.glob(f'{MERGED}/val/images/*'))print(f"\n\u2705 OCR dataset ready: {tc} train, {vc} val \u2014 10 digit classes (0-9)")

In [ ]:
# Cell 5B: Train jersey_ocr_universal_v5from ultralytics import YOLOMODEL_NAME = 'jersey_ocr_universal_v5'DATA_PATH = '/content/ocr_merged/data.yaml'# Guardassert os.path.exists(DATA_PATH), "\033[91m\u274c No OCR data \u2014 run Cell 5A first\033[0m"tc = len(glob.glob('/content/ocr_merged/train/images/*'))assert tc >= 100, f"\033[91m\u274c Need 100+ training images, have {tc}\033[0m"print(f"\u2705 {tc} training images ready")# Resume from checkpoint if existsresume_path = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/last.pt'if os.path.exists(resume_path) and os.path.getsize(resume_path) > 1024 * 1024:    print("Resuming from Drive checkpoint...")    model = YOLO(resume_path)    model.train(resume=True)else:    model = YOLO('yolov8m.pt')    model.train(        data=DATA_PATH,        epochs=120,        imgsz=640,        batch=DEFAULT_BATCH,        name=MODEL_NAME,        device=0,        patience=25,        save_period=5,        amp=True,        cache=True,        fliplr=0.0,   # Don't flip digits horizontally        mosaic=0.5,        degrees=5.0,        scale=0.3,    )# Save checkpoint to Driveckpt_dir = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}'os.makedirs(ckpt_dir, exist_ok=True)for wt in ['best.pt', 'last.pt']:    paths = sorted(glob.glob(f'runs/detect/{MODEL_NAME}*/weights/{wt}'))    if paths:        shutil.copy2(paths[-1], f'{ckpt_dir}/{wt}')torch.cuda.empty_cache()gc.collect()print(f"\n\u2705 {MODEL_NAME} training complete!")

In [ ]:
# Cell 5C: Save jersey_ocr_universal_v5.pt to DriveMODEL_NAME = 'jersey_ocr_universal_v5'paths = sorted(glob.glob(f'runs/detect/{MODEL_NAME}*/weights/best.pt'))if not paths:    alt = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/best.pt'    if os.path.exists(alt):        paths = [alt]assert paths and os.path.exists(paths[-1]), f"\033[91m\u274c No trained model \u2014 run Cell 5B first\033[0m"src = paths[-1]dst = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.pt'shutil.copy2(src, dst)sz = os.path.getsize(dst) / 1024 / 1024# Validate mAP50try:    m = YOLO(src).val()    map50 = m.box.map50    print("=" * 50)    print(f"\u2705 {MODEL_NAME}.pt SAVED TO DRIVE")    print(f"Location: {DRIVE_SAVE_DIR}/")    print(f"File size: {sz:.1f}MB")    print(f"mAP50: {map50:.3f}")    print("=" * 50)except Exception:    print("=" * 50)    print(f"\u2705 {MODEL_NAME}.pt SAVED TO DRIVE")    print(f"Location: {DRIVE_SAVE_DIR}/")    print(f"File size: {sz:.1f}MB")    print("=" * 50)

## SECTION 6: PLAYER DETECTORTrain a universal player detector combining basketball, football, and lacrosse datasets.**Verified datasets:**- `augmented-startups/football-player-detection-kucab` v1 (1,232 images)- `roboflow-universe-projects/basketball-players-fy4c2` v1- `ryseai/lacrosse-object-detection` v1- Alternative: `ai-in-sports/football-player-identification` v4 (5,174 images)Output: `player_detector_v5.pt`

In [ ]:
# Cell 6A: Download player detection datasetsfrom roboflow import Roboflowassert ROBOFLOW_API_KEY, "\033[91m\u274c Set ROBOFLOW_API_KEY in Colab Secrets\033[0m"rf = Roboflow(api_key=ROBOFLOW_API_KEY)def safe_download(ws, proj, ver, name):    for v in [ver] + [i for i in range(1, 6) if i != ver]:        try:            ds = rf.workspace(ws).project(proj).version(v).download('yolov8')            count = len(glob.glob(f'{ds.location}/train/images/*'))            print(f"\u2705 {name} v{v}: {count} images")            return ds        except Exception as e:            print(f"  v{v} failed: {e}")    print(f"\033[91m\u274c {name}: all versions failed\033[0m")    return Noneprint("=== Downloading Player Detection Datasets ===")ds_fb_players = safe_download('augmented-startups', 'football-player-detection-kucab', 1, 'football_players')ds_bb_players = safe_download('roboflow-universe-projects', 'basketball-players-fy4c2', 1, 'basketball_players')ds_lax_players = safe_download('ryseai', 'lacrosse-object-detection', 1, 'lacrosse_players')# === Merge into single player dataset ===print("\nMerging player datasets...")MP = '/content/player_merged'for s in ['train', 'val']:    os.makedirs(f'{MP}/{s}/images', exist_ok=True)    os.makedirs(f'{MP}/{s}/labels', exist_ok=True)for ds in [ds_fb_players, ds_bb_players, ds_lax_players]:    if not ds:        continue    loc = ds.location    dn = os.path.basename(loc)    for sp in ['train', 'valid', 'val', 'test']:        idir = f'{loc}/{sp}/images'        ldir = f'{loc}/{sp}/labels'        if not os.path.exists(idir):            continue        target = 'val' if sp in ['valid', 'val', 'test'] else 'train'        for img_path in glob.glob(f'{idir}/*'):            fn = f'{dn}_{os.path.basename(img_path)}'            shutil.copy2(img_path, f'{MP}/{target}/images/{fn}')            lbl = os.path.join(ldir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')            if os.path.exists(lbl):                # Remap all classes to 0 (player)                with open(lbl) as f:                    lines = ['0 ' + ' '.join(l.strip().split()[1:]) for l in f if len(l.strip().split()) >= 5]                if lines:                    with open(f'{MP}/{target}/labels/{os.path.splitext(fn)[0]}.txt', 'w') as f:                        f.write('\n'.join(lines) + '\n')# Write data.yamldata_yaml = {'path': MP, 'train': 'train/images', 'val': 'val/images', 'nc': 1, 'names': {0: 'player'}}with open(f'{MP}/data.yaml', 'w') as f:    yaml.dump(data_yaml, f)tc = len(glob.glob(f'{MP}/train/images/*'))vc = len(glob.glob(f'{MP}/val/images/*'))print(f"\n\u2705 Player dataset ready: {tc} train, {vc} val \u2014 1 class (player)")

In [ ]:
# Cell 6B: Train player_detector_v5from ultralytics import YOLOMODEL_NAME = 'player_detector_v5'DATA_PATH = '/content/player_merged/data.yaml'# Guardassert os.path.exists(DATA_PATH), "\033[91m\u274c No player data \u2014 run Cell 6A first\033[0m"tc = len(glob.glob('/content/player_merged/train/images/*'))assert tc >= 50, f"\033[91m\u274c Need 50+ training images, have {tc}\033[0m"print(f"\u2705 {tc} training images ready")# Resume from checkpoint if existsresume_path = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/last.pt'if os.path.exists(resume_path) and os.path.getsize(resume_path) > 1024 * 1024:    print("Resuming from Drive checkpoint...")    model = YOLO(resume_path)    model.train(resume=True)else:    model = YOLO('yolov8m.pt')    model.train(        data=DATA_PATH,        epochs=100,        imgsz=640,        batch=DEFAULT_BATCH,        name=MODEL_NAME,        device=0,        patience=25,        save_period=5,        amp=True,        cache=True,    )# Save checkpoint to Driveckpt_dir = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}'os.makedirs(ckpt_dir, exist_ok=True)for wt in ['best.pt', 'last.pt']:    paths = sorted(glob.glob(f'runs/detect/{MODEL_NAME}*/weights/{wt}'))    if paths:        shutil.copy2(paths[-1], f'{ckpt_dir}/{wt}')torch.cuda.empty_cache()gc.collect()print(f"\n\u2705 {MODEL_NAME} training complete!")

In [ ]:
# Cell 6C: Save player_detector_v5.pt to DriveMODEL_NAME = 'player_detector_v5'paths = sorted(glob.glob(f'runs/detect/{MODEL_NAME}*/weights/best.pt'))if not paths:    alt = f'{DRIVE_CHECKPOINTS}/{MODEL_NAME}/best.pt'    if os.path.exists(alt):        paths = [alt]assert paths and os.path.exists(paths[-1]), f"\033[91m\u274c No trained model \u2014 run Cell 6B first\033[0m"src = paths[-1]dst = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.pt'shutil.copy2(src, dst)sz = os.path.getsize(dst) / 1024 / 1024try:    m = YOLO(src).val()    map50 = m.box.map50    print("=" * 50)    print(f"\u2705 {MODEL_NAME}.pt SAVED TO DRIVE")    print(f"Location: {DRIVE_SAVE_DIR}/")    print(f"File size: {sz:.1f}MB")    print(f"mAP50: {map50:.3f}")    print("=" * 50)except Exception:    print("=" * 50)    print(f"\u2705 {MODEL_NAME}.pt SAVED TO DRIVE")    print(f"Location: {DRIVE_SAVE_DIR}/")    print(f"File size: {sz:.1f}MB")    print("=" * 50)

## SECTION 7: FINAL SUMMARYCheck all models and print final status.

In [ ]:
# Cell 7A: Check Drive folder and list all saved modelsprint(f"\nChecking {DRIVE_SAVE_DIR}...\n")all_files = sorted(glob.glob(f'{DRIVE_SAVE_DIR}/*.*'))if all_files:    total_size = 0    for f in all_files:        sz = os.path.getsize(f) / 1024 / 1024        total_size += sz        print(f"  {os.path.basename(f):45s} {sz:8.1f}MB")    print(f"\n  Total: {len(all_files)} files, {total_size:.1f}MB")else:    print("  No files found in Drive folder")    print(f"\033[91m\u274c No models saved yet \u2014 run Sections 3-6\033[0m")

In [ ]:
# Cell 7B: Print complete status tableALL_MODELS = [    'videomae_basketball_v5.zip',    'videomae_football_v5.zip',    'videomae_lacrosse_v5.zip',    'outcome_classifier_basketball_v5.pt',    'outcome_classifier_football_v5.pt',    'outcome_classifier_lacrosse_v5.pt',    'jersey_ocr_universal_v5.pt',    'player_detector_v5.pt',]print("=" * 50)print("V5 TRAINING COMPLETE")print("=" * 50)saved = 0trained_not_saved = 0missing = 0saved_list = []missing_list = []for mn in ALL_MODELS:    dp = f'{DRIVE_SAVE_DIR}/{mn}'    if os.path.exists(dp):        sz = os.path.getsize(dp) / 1024 / 1024        saved += 1        saved_list.append(f"\u2705 {mn} \u2014 {sz:.1f}MB")    else:        # Check if trained but not saved (in checkpoints or runs/)        base = mn.replace('.pt', '').replace('.zip', '')        ckpt = f'{DRIVE_CHECKPOINTS}/{base}/best.pt'        local_paths = glob.glob(f'runs/detect/{base}*/weights/best.pt') + glob.glob(f'runs/classify/{base}*/weights/best.pt')        if os.path.exists(ckpt) or local_paths:            trained_not_saved += 1            missing_list.append(f"\u26a0\ufe0f  {mn} \u2014 TRAINED NOT SAVED (run save cell)")        else:            missing += 1            missing_list.append(f"\u274c {mn} \u2014 MISSING")print(f"\nSAVED TO DRIVE ({saved}/8):")for line in saved_list:    print(f"  {line}")if missing_list:    print(f"\nNOT YET COMPLETE:")    for line in missing_list:        print(f"  {line}")print()print("=" * 50)if saved == 8:    print("\u2705 ALL 8 MODELS SAVED!")    print()    print("NEXT STEP: Download all files from Google Drive:")    print(f"  {DRIVE_SAVE_DIR}/")    print("and move them to your reelapp folder.")    print("Then send to Claude for integration prompt.")elif saved >= 6:    print(f"\u26a0\ufe0f {saved}/8 models saved \u2014 almost done!")    print("Run the missing save cells to complete.")else:    print(f"\u274c {saved}/8 models saved \u2014 run remaining training cells.")print("=" * 50)